## **2. Preprocess and Prepare Data:**

### **b. Understanding The Data by Visualization:**

We will use the "House Sales in King County, USA" dataset from Kaggle.
<br>Kaggle dataset link:
<br>https://www.kaggle.com/datasets/harlfoxem/housesalesprediction

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import display
import numpy as np
import os
import math

In [ ]:
CURRENT_DIR = os.getcwd()
PARENT_DIR = os.path.dirname(CURRENT_DIR)
GET_CSV_FILE = os.path.join(PARENT_DIR, "03.Pandas/kc_house_data _rev.csv")

df_houses = pd.read_csv(GET_CSV_FILE)

Examining the raw data can reveal valuable insights, which may help you pre-process the data, making your machine learning models perform better.

In [ ]:
# View the first 10 rows in the data:

with pd.option_context('display.max_columns', None):
    display(df_houses.head(10))

### **1. Univariate Plots:**

#### **Histograms:**
Are a fast way to get an idea of the distribution of each feature is to look at histograms.

In [ ]:
# We will drop some columns. Keep in mind that histograms truly shine with continuous data:
hist_df = df_houses.copy()
columns_to_drop = ["id", "date", "zipcode", "bedrooms", "yr_renovated", "floors", "grade", "bathrooms", "condition", "view", "waterfront"]
hist_df.drop(columns_to_drop, axis=1, inplace=True)

In [ ]:
hist_df.hist(figsize=(15, 10))
plt.show()

Most of the data have an exponential distribution, such as price, sqft_living, sqft_lot, sqft_above, sqft_basement, sqft_living15, and sqft_lot15, and are skewed to the right; we have a lot of small values on the left and a few large ones to the right.
<br> While the lat and yr_built are skewed to the left, most of their values are clustered toward the higher end (to the right), with a few smaller values pulling the distribution down (to the left).
<br>What does this imply?

This implies:
- Mean > Median: The average gets pulled up by those few large values when the data is skewed to the right.
- Mean < Median: The average is dragged lower by those few small values when the data is skewed to the left.
- Outliers: The large values (skewed to the right) or smaller values (skewed to the left) may be outliers, which can distort models such as linear regression.
- Feature scaling matters: Algorithms might perform better if you normalize the data. This will be discussed in detail later.
- Log transformation helps: Applying a log transformation can make the data more symmetric, which often improves model performance. This will be discussed in detail later.

Some machine learning models can be affected when the features are heavily skewed:
- Linear regression: Skewed predictors → nonlinear or distorted fit → residuals not normal → inference (p-values, confidence intervals) unreliable.
- Logistic regression: Skewed predictors can make the log-odds relationship nonlinear, reducing accuracy.
- Gaussian Naive Bayes: Assumes each feature itself is normally distributed within each class. If features are skewed, the assumption is directly violated, leading to poor probability estimates.
<br> Scaling, normalizing, or log transformation are essential for the model to return accurate results.

While tree-based models such as decision trees, random forests, and gradient boosting machines don't care if the data is normal or not.
<br>They split based on thresholds, not distributional assumptions.
<br>Some deep learning models are robust to skewness if they’re given enough data and the features are properly scaled, normalized or transformed.

#### **Density Plots:**
Density plots can be used to get a quick idea of the distribution of each feature.

In [ ]:
numeric_cols = hist_df.columns
n = len(numeric_cols)

ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.kdeplot(data=hist_df, x=col, ax=axes[i], fill=True, color='skyblue')
    axes[i].set_title(f'Density Plot: {col}', fontsize=10)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Density')
    axes[i].tick_params(axis='x', rotation=45)

# Remove unused axes:
# here "i" is the last index from the first loop. It’s used to determine which
# subplot axes were actually filled, so that any extra unused axes can be removed.

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Density plots are used in machine learning to visualize the probability distribution of continuous variables with smooth curves. This will make it easier to detect:
- skewness,
- multimodality (it has more than one peak (mode)),
- and subtle patterns that histograms might miss.
  - Histograms are discrete: They chop data into bins, which can hide fine details if bins are too wide.
  - Density plots are continuous: They smooth the data, making it easier to see small bumps, skewness, or a long tail.
<br>They are especially valuable for exploratory data analysis, feature engineering, and model diagnostics.

#### **BoxPlots:**
Boxplots are powerful tools in the machine learning workflow.

<center><img src="images/boxplots.jpg" width="700" height="400"/><center>

Boxplots summarize the distribution of each feature by showing the median and the 25th and 75th percentiles. The whiskers show how the data is spread, and the dots outside of the whiskers are outlier values.

The Boxplots are used in:
1. **Outlier detection**, as shown in the above picture. The points beyond the whiskers.
2. **Feature Comparison:** You can use boxplots to compare the distribution of numerical features across different classes or labels.

In [ ]:
# Example:
df_scores = pd.DataFrame({
    "study_method": ["Group Studying"] * 10 + ["Solo Studying"] * 10,
    'exam_score': [82, 85, 88, 90, 84, 87, 83, 89, 91, 86,
                   75, 78, 74, 80, 77, 76, 79, 81, 73, 72]
})

# Boxplot
sns.boxplot(x="study_method", y="exam_score", data=df_scores)
plt.title("Exam Scores by Study Method")
plt.xlabel("Study Method")
plt.ylabel("Exam Score")
plt.show()

3. **Understanding Data Distribution:**
Boxplots provide a visual summary of the central tendency, spread, and skewness of the data. Unlike histograms, which require bin size selection, boxplots are compact and intuitive for side-by-side comparisons of multiple features.

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))

# Boxplot
sns.boxplot(x="sqft_living", data=df_houses, ax=ax[0])
ax[0].set_title("Boxplot")
ax[0].set_xlabel("Living Area")

# Histogram
sns.histplot(x="sqft_living", data=df_houses, ax=ax[1])
ax[1].set_title("Histogram")
ax[1].set_xlabel("Living Area")
ax[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

4. Model Interpretation and Debugging Post-training, boxplots can help inspect feature importances or prediction errors across different groups. For instance, if a model underperforms for a certain class, a boxplot of residuals by class can hint at issues such as:
- Data imbalance in classification:
If one class in the target has fewer samples, the model may not learn it well. The boxplot would show higher residuals for that class (plotting residuals vs target values).
- Heteroscedasticity in the regression model:
    - It means that the variance of errors (residuals) in a model is not constant across all levels of the independent variable(s).
    - For example, in predicting house prices:
        - For small houses, the prediction errors might be tightly clustered (low variance).
        - For large luxury houses, the errors might be much more spread out (high variance).
    - That difference in spread is heteroscedasticity.

Let's say you are working on a regression model to predict student exam scores. Students are divided into classes based on their study method: "Group Studying" and "Solo Studying". After training, you compute residuals (i.e., actual - predicted scores) and want to check if the model performs worse for one group than the other.

In [ ]:
# Simulated data
np.random.seed(42)
n = 50
df_study = pd.DataFrame({
    "actual_score": np.random.normal(75, 10, size=2*n),
    "predicted_score": np.concatenate([
        np.random.normal(75, 8, size=n),      # Group Studying: better model fit
        np.random.normal(70, 15, size=n)      # Solo Studying: worse predictions
    ]),
    "studying_type": ["Group Studying"] * n + ["Solo Studying"] * n
})

# Calculate residuals
df_study["residual"] = df_study["actual_score"] - df_study["predicted_score"]

# Plotting a Boxplot showing residuals by studying type
plt.figure(figsize=(8, 5))
sns.boxplot(x="studying_type", y="residual", data=df_study)
plt.title('Residuals by Studying Method')
plt.axhline(0, linestyle="--", color="red")
plt.ylabel("Residual (Actual - Predicted)")
plt.show()

- In the Group Studying boxplot, the residuals are relatively tight (small spread).
- In the Solo Studying boxplot, the residuals are much more spread out (larger variance).
- This difference in spread across groups means the variance of residuals is not constant — exactly the definition of heteroscedasticity

You can repeat the residual boxplot approach for any feature in your dataset, and you can also use the fitted/predicted values on the x-axis. The key point is that you need to divide (bin) the feature values or predicted values into groups before plotting.

Example
- Feature-based: Residuals by Year Built bins (1950–1960, 1960–1970, …).
- Prediction-based: Residuals by Predicted Value bins (50–60, 60–70, 70–80, …).